# GRPO reward hacking: a gamed sentiment reward vs. the same run with a KL penaltyRuns the full experiment on a Colab **A100**. Two GRPO runs on `Qwen/Qwen2.5-0.5B-Instruct`continuing IMDB review openings, rewarded by a real sentiment classifier:| run | KL penalty | expectation ||---|---|---|| `baseline` | none (`beta = 0.0`) | reward climbs, output quality collapses || `fixed_kl` | `beta` vs. frozen base policy | reward climbs more slowly, quality holds |**Everything is resumable.** Each run pushes weights, optimizer state, step count, metricsand generated samples to its Hugging Face model repo. If Colab disconnects, just reopenthis notebook in a fresh runtime and re-run the cells: training picks up from the lastHugging Face checkpoint, not from step 0.Repo: https://github.com/Rohanjain2312/grpo-reward-hacking-demo

## 1. Check the GPURuntime -> Change runtime type -> **A100 GPU** before running this.

In [ ]:
!nvidia-smi

## 2. InstallClones the project and installs it. Takes a couple of minutes on a fresh runtime.

In [ ]:
import os, subprocess, sysREPO_URL = "https://github.com/Rohanjain2312/grpo-reward-hacking-demo.git"WORKDIR = "/content/grpo-reward-hacking-demo"if not os.path.isdir(WORKDIR):    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)else:    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only"], check=False)os.chdir(WORKDIR)!pip install -q -e . 2>&1 | tail -2print("installed:", os.getcwd())

## 3. Hugging Face loginNever hardcode a token in this notebook. Two options, in order of preference:1. **Colab secret** (recommended): sidebar key icon -> add a secret named `HF_TOKEN`   holding a token with **write** access -> toggle notebook access on.2. Interactive login, which the cell falls back to.The token needs write access because the run pushes checkpoints to your model repos.

In [ ]:
import ostoken = Nonetry:    from google.colab import userdata    token = userdata.get("HF_TOKEN")    print("using HF_TOKEN from Colab secrets")except Exception:    passfrom huggingface_hub import login, whoamiif token:    login(token=token)else:    login()  # interactive promptos.environ["HF_TOKEN"] = token or ""print("logged in as:", whoami()["name"])

## 4. The configuration being runThese are the values the reported results use. `beta` is the entire difference betweenthe two runs.

In [ ]:
from grpo_demo.config import CONFIGSimport jsonfor name, cfg in CONFIGS.items():    d = cfg.to_dict()    print(f"--- {name} ---")    print(json.dumps({k: d[k] for k in        ["policy_model","reward_model","total_steps","prompts_per_step","group_size",         "max_new_tokens","learning_rate","beta","eval_every","full_ckpt_every","hf_repo"]}, indent=2))

## 5. Baseline run (no KL) -- the one that gets gamedAbout 12 min for 100 steps on an A100 (measured: ~7 s/step, plus an eval every 10steps and a full checkpoint every 40). Re-running this cell after a disconnect resumesfrom the last Hugging Face checkpoint instead of restarting.**The published runs stopped at step 80 of 100** (the training account ran out ofHugging Face Jobs credits). Because `latest.json` in each model repo points at step 80,running this cell now continues from step 80 to 100 rather than starting over.

In [ ]:
!python -m grpo_demo.train --config baseline

## 6. Fixed run (KL penalty)Same seed, same prompts, same schedule -- only `beta` differs (0.0 vs 0.1). Same runtime and same step-80 resume behaviour as the cell above.

In [ ]:
!python -m grpo_demo.train --config fixed_kl

## 6b. Reward-capping run (the alternative mitigation)The second way to stop the chase: clip the sentiment score at 0.9 instead of penalisingdivergence. Once every completion in a group clears the cap their rewards are identical,the group-relative advantage goes to zero, and there is nothing left to optimise.This arm starts from scratch (no prior checkpoint), so it is the full ~12 min.

In [ ]:
!python -m grpo_demo.train --config fixed_cap

## 7. Independent quality judgingLoads Qwen2.5-7B-Instruct and rates every logged completion 1-5 for writing qualityagainst a fixed rubric that explicitly excludes sentiment, so the metric staysindependent of the training reward. Results are pushed to `logs/judge.jsonl` in eachmodel repo.

In [ ]:
!python -m grpo_demo.judge --runs baseline fixed_kl fixed_cap

## 8. FiguresWrites article-ready PNGs to `figures/`.

In [ ]:
!python -m grpo_demo.figuresfrom IPython.display import Image, displayfor f in ["fig1_reward_hacking_diagnosis.png", "fig2_reward_vs_step.png",          "fig3_quality_vs_step.png", "fig4_secondary_metrics.png",          "fig5_sample_completions.png"]:    display(Image(f"figures/{f}"))

## 9. Read the actual outputsThe point of the demo. Same prompt, same step, both policies.

In [ ]:
import jsonfrom pathlib import Pathdef load(run):    p = Path(f"results/{run}/judge.jsonl")    if not p.exists():        p = Path(f"results/{run}/samples.jsonl")    return [json.loads(l) for l in open(p) if l.strip()] if p.exists() else []RUNS = [("BASELINE (no KL)", "baseline"), ("FIXED (KL)", "fixed_kl"), ("FIXED (cap)", "fixed_cap")]loaded = [(tag, load(r)) for tag, r in RUNS]loaded = [(tag, rows) for tag, rows in loaded if rows]steps = sorted({r["step"] for _, rows in loaded for r in rows})for step in (steps[0], steps[-1]):    print("=" * 100)    print(f"STEP {step}")    for tag, rows in loaded:        sel = sorted([r for r in rows if r["step"] == step], key=lambda r: r["opening"])        if not sel:            continue        row = sel[0]        print(f"\n  [{tag}] reward={row['reward']:.3f}"              + (f" judge={row['judge_score']:.2f}" if "judge_score" in row else ""))        print(f"  prompt: {row['opening'][:110]}")        print(f"  ->      {row['completion'][:400]}")    print()

## Resuming after a disconnectNothing special to do. `grpo_demo.train` calls `HubCheckpointer.find_latest()`, whichreads `latest.json` from the run's model repo on the Hub, downloads that step-numberedcheckpoint (weights + optimizer state + training state), replays the LR schedule to thatstep and continues. Metrics and samples logged before the disconnect are pulled back downtoo, so the curves stay continuous.To start a run over from scratch, delete `latest.json` from the model repo.